# Burstormer Teacher → NAFNet Student：知識蒸餾專案

## 1. 專案目的與目前進度（Project Purpose & Status）

本專案的目標是進行 **output-level Knowledge Distillation（輸出層知識蒸餾，KD）**：使用預訓練且凍結的 **Burstormer Teacher** 處理完整多幀 burst；使用可訓練的 **NAFNet Student** 處理同一筆資料的單張 noisy reference frame。Student 的輸出同時與 clean reference 計算 supervised loss，並與 Teacher 輸出計算 KD loss：

```text
L_total = L_GT + lambda_kd × L_KD
L_GT    = L1(student_output, clean_reference)
L_KD    = L1(student_output, teacher_output)
```

目前已完成的是**程式流程驗證（pipeline/integration validation）**，不是正式模型評估或公平實驗：

- Open Images V6 的下載程式已在本專案執行；小型下載最多 10 張，實際有 9 張符合資料尺寸條件。
- Dataset 能從 clean source image 即時生成 8-frame burst、noisy frames、對應 sigma map、單幀 Student input 和 clean target。
- 使用實際 Burstormer checkpoint 與 NAFNet，在 CPU 上完成一個 KD 更新步驟。
- 使用 9 張可用圖片完成 1 epoch（7 train、2 validation），並保存 Student checkpoint。

成功執行表示模型可以載入、資料張量可供模型使用、loss 和反向傳播可執行；**不代表**模型去噪品質良好、noise protocol 已符合 Teacher 預訓練分佈、實驗公平，或已達成論文結果。正式 20K 樣本定義、獨立 test set、PSNR/SSIM 評分和 post-processing 尚未完成。

## 2. 模型與資料流（Model / Data Flow）

```text
Open Images V6 clean photograph
    │
    └── NAFNetKDBurstDataset：抽取共同區域，產生 8 張對齊 clean frames
          │
          ├── white level / read noise / shot noise 隨機取樣並合成 noisy burst
          ├── 使用同一筆 noise parameters 計算逐幀 sigma map
          │
          ├── noisy burst + sigma map [8, 6, H, W] ──→ Frozen Burstormer Teacher
          ├── noisy reference frame 0 [3, H, W] ─────→ Trainable NAFNet Student
          └── clean reference frame 0 [3, H, W] ─────→ Ground-truth target

Teacher output ─┐
                ├── L_KD；Student output 與 clean target 計算 L_GT
Student output ─┘
```

DataLoader 加入 batch dimension 後，Teacher 輸入是 `[1, 8, 6, H, W]`，Student 輸入與兩個模型輸出是 `[1, 3, H, W]`。目前本地 Burstormer forward 有 batch size 1 的限制，所以正式訓練的 DataLoader 也設為 batch size 1。

Clean 影像是 source data；burst/noise 是 Dataset **讀取時即時合成（on-the-fly）**。目前不會在磁碟上產生一份 processed noisy images 資料集，也不需要分開執行「先做 clean reference」和「再加 noisy burst」兩支 preprocessing 程式。

## 3. 專案目錄（Project Structure）

```text
3rd_KD_Completed/
├── README.ipynb                        # 本專案主要使用與交接指南
├── requirements.txt                    # PyTorch 以外的共用依賴
├── train_kd.py                         # KD 訓練入口（含 train/validation）
├── run_kd_smoke.py                     # 隨機假資料的一步 smoke test
├── BURSTORMER_INTEGRATION_NOTES.md      # Teacher 整合狀態、限制和待確認事項
├── checkpoints_kd/                     # KD Student 輸出；本機試跑產物
├── weights/
│   └── Burstormer.ckpt                 # 預訓練 Teacher 權重
├── data/
│   ├── raw/                            # FiftyOne 下載的 Open Images（可重新下載）
│   ├── processed/                      # 預留；現行程式沒有使用
│   ├── download_open_images.py         # Open Images V6 下載器
│   ├── verify_kd_dataset.py            # 小批次資料生成與張量檢查
│   ├── nafnet_kd_dataset.py            # 正式 KD 的 burst/noise 即時生成
│   ├── synthetic_smoke_dataset.py      # smoke test 專用 dummy data
│   ├── burst_dataset.py                # 舊 .npz loader，未接入 train_kd.py
│   └── README.ipynb
├── losses/
│   ├── distillation.py                 # GT L1 與 output KD L1
│   └── README.ipynb
├── models/
│   ├── FACTORY_CONTRACT.md              # 模型 factory 和 tensor 契約
│   ├── burstormer/
│   │   ├── Network.py                  # Burstormer architecture
│   │   ├── burstormer_teacher.py       # Teacher adapter
│   │   └── README.ipynb
│   └── nafnet/
│       ├── nafnet_arch.py              # 本專案內的 NAFNet architecture
│       ├── NAFNet_UPSTREAM_LICENSE.txt  # architecture 對應授權
│       ├── nafnet_student.py           # 正式 Student adapter
│       ├── demo_student.py             # smoke regression 用小模型
│       └── README.ipynb
└── utils/
    └── smoke_training.py               # smoke test 專用 one-step helper
```

上表列出程式碼與必要文件。`.venv/`、下載資料、checkpoint 等產物不屬於可攜式程式碼本體。

## 4. Windows 安裝與執行（Quick Start）

### 4.1 解壓 ZIP 並建立 Python 環境

建議使用 Python 3.11 或 3.12。先將 ZIP 解壓到任意資料夾，PowerShell 切換到 `3rd_KD_Completed` 專案根目錄，然後建立新的虛擬環境：

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
```

若 PowerShell 阻擋虛擬環境啟動腳本，可不啟動環境，改用完整路徑呼叫 `.\.venv\Scripts\python.exe`。

### 4.2 安裝 PyTorch 和其餘依賴

請先依硬體安裝彼此相容的 `torch` 和 `torchvision`：

- **純 CPU 電腦（如本專案作者筆電）**：`python -m pip install torch torchvision`
- **NVIDIA GPU 電腦**：前往 [PyTorch Get Started](https://pytorch.org/get-started/locally/)，選 Windows、pip、Python 及適合 GPU driver 的 CUDA 組合，執行頁面提供的 PyTorch／torchvision 指令。

接著在專案根目錄安裝其餘依賴：

```powershell
python -m pip install -r requirements.txt
```

驗證套件和 CUDA：

```powershell
python -c "import torch; print('torch:', torch.__version__); print('CUDA build:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available())"
```

使用 `--device cuda` 前，必須確認 `CUDA available: True`。CPU-only PyTorch 遇到 CUDA-only 指令會報 `Torch not compiled with CUDA enabled`。ZIP 不應包含作者的 `.venv`；每位使用者都必須在自己的硬體環境重建並安裝套件。依賴清單說明見 [requirements.txt](./requirements.txt)。

### 4.3 下載少量 Clean Images 並驗證資料

`data/download_open_images.py` 使用 FiftyOne 從網路下載 Open Images V6 train split。從專案根目錄執行：

```powershell
python -m data.download_open_images --max-samples 10
python -m data.verify_kd_dataset --num-samples 10
```

下載圖片目錄為 `data/raw/open-images-v6/train/data/`。下載需要網路連線，FiftyOne 會同時取得資料集 metadata；下載數字是**最多原始圖片數**，不是已完成的 burst patch 數。現行 Dataset 只接受該目錄第一層內 JPG/JPEG/PNG，並略過小於 `640×640` 或無法讀取的圖片。本次 10 張中有 9 張可用。

驗證指令會讀取可用原圖並生成少量完整 sample，檢查輸入輸出形狀、finite values、reference 是否等於 burst 第 0 幀、sigma channels 是否一致。它不會訓練模型，也不會保存 noisy bursts。

### 4.4 本機 CPU 小型 Training Check

沒有 GPU 也能確認完整程式訓練路徑。已有成功紀錄的指令：

```powershell
python train_kd.py --device cpu --epochs 1
```

本次輸出為 9 張 usable images、7 train／2 validation，完成 1 epoch，checkpoint 寫入 `checkpoints_kd/nafnet_kd_best.pt`。這是小型流程測試，不能視為正式 NAFNet 訓練、模型品質分數或公平比較。CPU 訓練較慢，正式模型訓練建議交由相容的 GPU 環境。

### 4.5 組員 GPU 執行

組員收到 ZIP 後自行建立環境、安裝相容的 GPU PyTorch，再重新下載資料，先執行小樣本確認後才擴大下載：

```powershell
python -m data.download_open_images --max-samples 10
python -m data.verify_kd_dataset --num-samples 10
python train_kd.py --device cuda --epochs 1
```

確定能載入 Teacher、進行 GPU forward/backward 並寫出 checkpoint 後，再由組員依協議調整正式樣本規模和 epochs。訓練參數可以透過 `--help` 查看；`lambda_kd=0.1`、learning rate `1e-4`、epochs `50` 是目前程式預設，不代表已由實驗選定。

## 5. Smoke Test 與正式資料 Training 的差別

| 項目 | Smoke Test | KD training |
|---|---|---|
| 入口 | `run_kd_smoke.py` | `train_kd.py` |
| Dataset | `synthetic_smoke_dataset.py` 隨機生成 | `nafnet_kd_dataset.py` 讀取 Open Images clean source |
| sigma channels | 固定 placeholder `0.01` | 使用此次生成的 `sigma_read`、`sigma_shot` 計算逐像素 map |
| clean target | 與 noisy input 無關的 toy tensor | 由同一 burst 的 clean reference frame 得到 |
| Student | 可選 DemoStudent 做 quick regression；亦可選 NAFNet | NAFNet |
| 目的 | 驗證模型介面、loss、gradient 和 optimizer step | 實際跑 train/validation loop |
| 可否代表去噪品質 | 不可 | 仍需獨立 test set 和 metrics 才可評估 |

Smoke test 成功並不表示 sigma estimate 正確；它使用 placeholder。真實資料路徑則會根據每筆生成噪聲的參數計算 map，但仍需確認 noise distribution、影像值域、gamma／white-level 處理與 Teacher 預訓練流程相符。

## 6. 20K 的定義與限制（Important）

論文提及的 20K 是 synthetic burst patches／samples 的計數，不應直接等同「下載 20,000 張 Open Images 原始照片」。本專案目前 downloader 的 `--max-samples` 只控制原圖下載上限；Dataset 長度是符合尺寸條件的 source images 數，每次讀取再生成一組 sample。訓練時每次 access 會重新抽 crop/noise，因此不同 epoch 的實際樣本可以不同。

`train_kd.py` 目前將可用圖片索引切成 80% train、20% validation，batch size 固定為 1、預設 50 epochs。例如有 20,000 張**可用**來源圖時，單次 split 約為 16,000 train、4,000 validation；但下載 20,000 張不保證全數滿足 640×640，故 train 數可能更低。

目前尚無固定建立「恰好 20K bursts」的 preprocessing job／manifest，也沒有獨立 test split。正式開跑前要與組員確認：

1. 20K 指 train-only patches，還是 train+validation 的總數？
2. 20K 是不同來源圖數，還是可重複從較少來源圖抽 crop/noise 形成的 sample 數？
3. 需要幾個 split、比例為何、test 是否使用論文指定噪聲參數？
4. NAFNet baseline 和 KD 是否必須逐筆使用相同 crop 與 noise realization？若需要，目前只有共用生成邏輯，尚未保證逐筆相同；需建立共用 seed/manifest 或快取。
5. epochs、lambda、learning rate 及是否使用 pretrained NAFNet 初始化是否已定案？

因此目前可以交接的是可執行的 training pipeline，而不是已定稿的 20K experiment protocol。

## 7. Checkpoint、結果與 Lightning Warning

- `weights/Burstormer.ckpt` 是固定的預訓練 Teacher 權重。程式從中載入 Teacher，將它設為 eval/frozen；訓練不更新此 checkpoint。
- `checkpoints_kd/nafnet_kd_best.pt` 是 Student training 依 validation total loss 儲存的 best checkpoint。本次檔案來自 9 張 usable images、1 epoch CPU test，只是 pipeline artifact，不要當成正式模型。
- 載入 Teacher checkpoint 時 Lightning 顯示舊 checkpoint migration warning，但模型在本機成功載入、forward 並完成更新。測試沒有把 Teacher checkpoint 改寫。
- `train_kd.py` 印出 GT/KD/total train 和 validation losses；**尚未計算 PSNR、SSIM、LPIPS，也沒有正式 test score 或 post-processing runner**。

## 8. ZIP 交接與安全清理

建議 ZIP 包含程式碼、README、requirements、`weights/Burstormer.ckpt`（或另外提供取得方式），以及必要的 upstream license。通常可排除以下本機／大型產物：

| 項目 | 本機用途 | ZIP 建議 |
|---|---|---|
| `.venv/` | Python 套件環境，本機約 1.4 GB | 排除；組員自行重建 |
| `data/raw/` | 10 張 Open Images 和 FiftyOne metadata，本機約 612 MB | 排除；組員可重新下載 |
| `data/processed/` | 目前空白預留資料夾，訓練未使用 | 排除 |
| `checkpoints_kd/` | 9 張資料／1 epoch 的試跑 Student checkpoint，約 335 MB | 排除，不是正式權重 |
| `__pycache__/`、`.ipynb_checkpoints/` | Python／Notebook 快取 | 排除 |
| `weights/Burstormer.ckpt` | Teacher 權重，約 36 MB | 保留或確保組員另行取得 |

本機若要清理，只刪除明確的資料目錄，不要刪掉整個 `data/`：

```powershell
Remove-Item -Recurse -Force .\data\raw
Remove-Item -Recurse -Force .\checkpoints_kd
```

這些命令會刪除本機測試圖片和該次試跑權重；若還要重看或重現測試，可先保留。`.gitignore` 已忽略 `.venv/`、`data/raw/`、`data/processed/` 與 `checkpoints_kd/`。ZIP 前仍應檢查壓縮檔內容，不要把大型資料或本機環境夾進去。

## 9. 檔案責任與延伸閱讀

- `train_kd.py`：正式 KD train/validation entry，建立 Dataset、凍結 Teacher、更新 NAFNet、保存 best checkpoint。
- `run_kd_smoke.py`、`utils/smoke_training.py`：單 batch smoke、確認梯度路徑。
- `data/download_open_images.py`：FiftyOne clean source downloader。
- `data/verify_kd_dataset.py`：取少量實際圖片檢查生成資料。
- `data/nafnet_kd_dataset.py`：burst crop/noise synthesis、sigma map、Student input 與 target。
- `data/synthetic_smoke_dataset.py`：只供 dummy smoke data。
- `data/burst_dataset.py`：舊 `.npz` loader，沒有被目前正式 trainer 呼叫。
- `models/nafnet/`：architecture、adapter、DemoStudent 與 license。
- `models/burstormer/`：Teacher architecture 和 checkpoint adapter。
- `losses/distillation.py`：兩種 L1 loss。

詳細資料契約見 [data README](./data/README.ipynb)、[Loss README](./losses/README.ipynb)、[Burstormer README](./models/burstormer/README.ipynb)、[NAFNet README](./models/nafnet/README.ipynb)、[factory contract](./models/FACTORY_CONTRACT.md) 與 [Burstormer integration notes](./BURSTORMER_INTEGRATION_NOTES.md)。

## 10. 正式評估仍待完成（Not Yet Implemented）

1. 確認 20K 的計數單位、下載策略、split 和可重現樣本定義。
2. 與 NAFNet baseline 協調共用的資料生成條件／sample manifest。
3. 查證 synthetic noise、noise parameter 定義、gamma／white-level 與值域是否符合 Teacher 預訓練輸入。
4. 定義獨立 test dataset、固定測試噪聲條件、post-processing 和 PSNR／SSIM（或其他）評估。
5. 在組員 GPU 上重新安裝依賴並做小型 CUDA validation，再開始正式訓練。

完成上述事項前，不要把本機一個 epoch 的 loss 或 checkpoint 當成模型結論。

## 11. 常見問題（Troubleshooting）

### `Torch not compiled with CUDA enabled`

這表示目前安裝的是 CPU-only PyTorch，卻用 `--device cuda` 要求 GPU。作者的筆電只有 CPU，故本機使用 `--device cpu`；組員需在自己的 NVIDIA GPU 環境重新安裝相容的 CUDA PyTorch 和 torchvision。單純把 `--device` 改成 cuda，不會替 CPU PyTorch 增加 CUDA 能力。

### Open Images 下載成功但 usable images 較少

FiftyOne 的 `max-samples` 是下載上限；Dataset 還會忽略圖片短邊不足 640 pixel 的影像，以及不能解碼的檔案。先用 `verify_kd_dataset` 查看可用數量。正式 sample count 要以 Dataset 能讀取且符合尺寸條件的數量為準，不能只看 downloader 的 record count。

### `Burstormer.ckpt` 載入時有 Lightning warning

現有舊 checkpoint 在目前 Lightning 中會出現 migration/callback state warning。這些 warning 曾伴隨成功的 CPU model load、forward 和 one-epoch training。若日後 warning 變成 exception，需再檢查組員的 Lightning version、checkpoint 完整性及模型來源版本；不可在未備份前任意執行會重寫 checkpoint 的 upgrade command。

### Dataset shape 成功，是否代表噪聲參數已確認？

不是。`verify_kd_dataset` 確認資料欄位、shape、finite values 和 frame pairing；它不會驗證學術 noise protocol，也不會比較 Teacher 的 pretrained distribution。Noise model、單位、gamma/white-level 和數值範圍須另行核實。

### Loss 降低，是否代表測試分數提升？

不是。Training/validation loss 是訓練 loop 的像素 L1 objective；目前沒有 test split、PSNR/SSIM 評分和 post-processing runner。即使 loss 下降，也不可稱為 benchmark score 或與 baseline 公平比較的結果。

## 12. 執行命令速查（Command Cheat Sheet）

以下命令假設 PowerShell 的目前目錄是 `3rd_KD_Completed`，且虛擬環境已建立：

```powershell
# 查詢 downloader、validator 和 trainer 可用的參數
python -m data.download_open_images --help
python -m data.verify_kd_dataset --help
python train_kd.py --help

# 建立 10 張小型 download set 並檢查全部可用 samples
python -m data.download_open_images --max-samples 10
python -m data.verify_kd_dataset --num-samples 10

# CPU 最小 training check
python train_kd.py --device cpu --epochs 1

# GPU 小型 training check（需 CUDA-enabled PyTorch）
python train_kd.py --device cuda --epochs 1

# 較大規模的下載需先確認磁碟空間及 20K 定義
python -m data.download_open_images --max-samples 20000
```

實際正式訓練的 `--max-samples`、epochs、split、noise settings 和 `lambda_kd` 不應只由這份 quick start 決定；先和組員確認使用相同的 data protocol。

## 13. ZIP 實際清理檢查清單

建立 ZIP 前建議手動確認：

1. ZIP 根目錄內包含 `train_kd.py`、`run_kd_smoke.py`、`requirements.txt`、`README.ipynb` 和各 package 程式檔，而非多包一層不明資料夾。
2. `data/` 內的 `.py` 和 README 有保留；`data/raw/` 可排除。
3. 不包含 `.venv/`、`.git/`、`__pycache__/`、`data/processed/` 空資料夾和小型 `checkpoints_kd/` 試跑 checkpoint。
4. Teacher 的 `weights/Burstormer.ckpt` 有包含，或 README 清楚說明由收件者另行取得的路徑。
5. NAFNet architecture 的上游 license 保留。
6. 收件者解壓後重新安裝依賴、重新下載 10 張圖片，再跑 validator 和 1 epoch check。

不要直接把整個工作資料夾壓縮後就傳送；本機 `.venv` 和 FiftyOne metadata 會讓 ZIP 過大，但它們不是程式碼必要依賴。另一方面，若傳送的訊息平台有檔案大小限制，Teacher checkpoint 也可能需分開傳送或另行提供。
